In [1]:
import pandas as pd


In [2]:
import os 

In [3]:
pip install langchain


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install langchain_google_genai


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install langchain_ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [7]:
from langchain_ollama import ChatOllama

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [9]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from dotenv import load_dotenv
import os

load_dotenv()  # o load_dotenv("llaves.env")

api_key = os.getenv("GOOGLE_API_KEY")

print(api_key[:5])  # prueba

AIzaS


In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [13]:
history = []

history.append(SystemMessage(content="""
Eres un asesor financiero digital especializado en ayudar a personas con bajos ingresos en Colombia.

Tu objetivo es:
- Analizar la situación financiera del usuario
- Dar recomendaciones prácticas y realistas
- Educar al usuario con lenguaje sencillo

Reglas:
- Usa máximo 4 frases
- No uses lenguaje técnico complejo
- Sé claro, directo y útil
- Prioriza evitar el sobreendeudamiento
- Da máximo 5 recomendaciones

Formato de respuesta:
1. Diagnóstico breve
2. Recomendaciones
3. Explicación sencilla
"""))

In [14]:
def construir_input_usuario(data):
    return f"""
    El usuario tiene la siguiente situación financiera:

    Ingresos mensuales: {data['ingresos']}
    Gastos fijos: {data['gastos_fijos']}
    Gastos variables: {data['gastos_variables']}
    Deudas: {data['deudas']}

    Genera recomendaciones financieras personalizadas.
    """

In [15]:
def encuesta_inicial():

    texto = input("Ingrese: ingresos, gastos fijos, gastos variables, deudas (separados por coma): ")

    try:
        ingresos, gastos_fijos, gastos_variables, deudas = map(int, texto.split(","))

        user_data = {
            "ingresos": ingresos,
            "gastos_fijos": gastos_fijos,
            "gastos_variables": gastos_variables,
            "deudas": deudas
        }

        # Construir prompt estructurado
        prompt = construir_input_usuario(user_data)

        history = []
        history.append(SystemMessage(content="""
        Eres un asesor financiero para personas con bajos ingresos.
        Da recomendaciones claras, simples y útiles en máximo 4 frases.
        """))

        history.append(HumanMessage(content=prompt))

        response = llm.invoke(history)

        print("\n--- RESULTADO ---\n")
        print(response.content)

    except:
        print("Error: asegúrate de ingresar números separados por comas.")

In [16]:
def analizar_usuario(data):
    ahorro = data["ingresos"] - (data["gastos_fijos"] + data["gastos_variables"])
    deuda_ratio = data["deudas"] / data["ingresos"]

    return ahorro, deuda_ratio

In [17]:
def encuesta_prueba():

    print("Inicio función")

    user_data = {
        "ingresos": 1500000,
        "gastos_fijos": 800000,
        "gastos_variables": 400000,
        "deudas": 300000
    }

    prompt = construir_input_usuario(user_data)
    print("Prompt construido")

    history = []
    history.append(SystemMessage(content="Eres un asesor financiero."))
    history.append(HumanMessage(content=prompt))

    print("Antes del invoke")

    try:
        response = llm.invoke(history)
        print("Después del invoke")
        print(response)
        print(response.content)
    except Exception as e:
        print("ERROR:", e)

In [18]:
encuesta_prueba()

Inicio función
Prompt construido
Antes del invoke
Después del invoke
content='Hola, gracias por compartir tu situación financiera. He analizado tus números y aquí está un resumen:\n\n*   **Ingresos mensuales:** $1,500,000\n*   **Gastos fijos:** $800,000\n*   **Gastos variables:** $400,000\n*   **Pagos de deudas mensuales:** $300,000\n*   **Total de gastos y pagos:** $800,000 + $400,000 + $300,000 = $1,500,000\n*   **Ahorro mensual actual:** $1,500,000 (Ingresos) - $1,500,000 (Gastos) = **$0**\n\n**Análisis Inicial:**\n\nActualmente, tus ingresos cubren exactamente tus gastos y pagos de deudas, sin dejar un excedente para ahorro o inversión. Aunque no estás en déficit, la falta de un colchón financiero puede generar vulnerabilidad ante imprevistos (gastos médicos, reparaciones, pérdida de empleo, etc.) y dificulta el logro de metas financieras a largo plazo.\n\n**Recomendaciones Financieras Personalizadas:**\n\nDada tu situación, las prioridades inmediatas son: **crear un excedente mens